# 03 - Model Training & Evaluation
## CSE-CIC-IDS2018 — XGBoost Multi-Class

**Pipeline:**
1. Train model induk (seluruh fitur) pada file_100 → Top-20 features
2. Train & test pada 4 ukuran dataset (100%, 75%, 50%, 25%) — split 80/20
3. Metrics: Accuracy, Precision, Recall, F1, ROC-AUC
4. Efficiency: Training time, Inference time, Model size, RAM
5. Visualisasi: Learning Curve, Confusion Matrix, Feature Importance

**Input:** `cleaned_*.pkl` dari Notebook 02

In [ ]:
# Install dependencies (run once per session)
import sys
!{sys.executable} -m pip install scikit-learn xgboost matplotlib seaborn psutil -q
print('✓ Dependencies installed')

In [ ]:
import pandas as pd
import numpy as np
import pickle, os, gc, time, sys, warnings
import psutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 10, 'figure.dpi': 120})

DATA_DIR = '../data/'
MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)

RANDOM_SEED = 42
TEST_SIZE = 0.20

DATASETS = {
    '100%': 'cleaned_100.pkl',
    '75%': 'cleaned_75.pkl',
    '50%': 'cleaned_50.pkl',
    '25%': 'cleaned_25.pkl'
}

# Models yang akan dibandingkan
MODELS = {
    'XGBoost': lambda n_classes: XGBClassifier(
        n_estimators=200, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softprob', num_class=n_classes,
        eval_metric='mlogloss', random_state=RANDOM_SEED,
        n_jobs=-1, tree_method='hist'),
    'Random Forest': lambda n_classes: RandomForestClassifier(
        n_estimators=200, max_depth=20, min_samples_split=5,
        random_state=RANDOM_SEED, n_jobs=-1),
    'SVM': lambda n_classes: SVC(
        kernel='rbf', C=10, gamma='scale',
        decision_function_shape='ovr', probability=True,
        random_state=RANDOM_SEED)
}

print(f'Data dir: {DATA_DIR}')
print(f'Model dir: {MODEL_DIR}')
print(f'Test size: {TEST_SIZE*100:.0f}%')
print(f'Random seed: {RANDOM_SEED}')
print(f'Models: {list(MODELS.keys())}')

## 1. Train Model Induk (Seluruh Fitur) → Top-20 Features

In [ ]:
# Load dataset terbesar sebagai referensi
with open(os.path.join(DATA_DIR, 'cleaned_100.pkl'), 'rb') as f:
    data_100 = pickle.load(f)

X_full = data_100['X']
y_full = data_100['y']
feature_names = data_100['feature_names']
label_mapping = data_100['label_mapping']

print(f'Dataset 100%: X={X_full.shape}, y={y_full.shape}')
print(f'Features: {len(feature_names)}')
print(f'Classes: {len(label_mapping)}')
print(f'Label mapping: {label_mapping}')

In [ ]:
# Train model induk XGBoost (seluruh fitur) untuk extract feature importance
print('Training model induk XGBoost (all features) for feature importance...')

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X_full, y_full, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_full)

n_classes = len(np.unique(y_full))
model_induk = MODELS['XGBoost'](n_classes)

start = time.time()
model_induk.fit(X_train_full, y_train_full)
train_time_induk = time.time() - start

print(f'Training time: {train_time_induk:.2f}s')
print(f'Train shape: {X_train_full.shape} | Test shape: {X_test_full.shape}')

In [ ]:
# Extract Top-20 Feature Importance
importances = model_induk.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('='*70)
print(f'{"TOP-20 FEATURES (Model Induk — All Features)":^70}')
print('='*70)
for i, row in feat_imp_df.head(20).iterrows():
    bar = '█' * int(row['importance'] * 100)
    print(f"  {i+1:>2d}. {row['feature']:35s} {row['importance']:.4f} {bar}")

# Save feature importance
feat_imp_df.to_csv(os.path.join(DATA_DIR, 'feature_importance_all.csv'), index=False)
top20_features = feat_imp_df.head(20)['feature'].tolist()
top15_features = feat_imp_df.head(15)['feature'].tolist()
top10_features = feat_imp_df.head(10)['feature'].tolist()

print(f'\nSaved: feature_importance_all.csv')
print(f'Top-20: {top20_features}')

## 2. Helper Functions

In [ ]:
def get_model_size(model, filepath):
    """Save model and return file size in MB."""
    with open(filepath, 'wb') as f:
        pickle.dump(model, f)
    size_mb = os.path.getsize(filepath) / (1024*1024)
    return size_mb

def get_memory_usage():
    """Return current process RAM usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024*1024)

def train_and_evaluate(X, y, model_name, model_factory, dataset_name, model_tag):
    """
    Train model, evaluate, return metrics dict.
    """
    # Split 80/20
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y)
    
    n_classes = len(np.unique(y))
    model = model_factory(n_classes)
    
    # Training
    mem_before = get_memory_usage()
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start
    mem_after = get_memory_usage()
    
    # Inference
    start = time.time()
    y_pred = model.predict(X_test)
    inference_time = time.time() - start
    # Normalize to 10k requests
    inference_per_10k = (inference_time / len(X_test)) * 10000
    
    # Probabilities for ROC-AUC
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)
    else:
        y_prob = None
    
    # Metrics
    acc = accuracy_score(y_test, y_pred) * 100
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100
    try:
        if y_prob is not None:
            roc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted') * 100
        else:
            roc = 0.0
    except:
        roc = 0.0
    
    # Model size
    model_path = os.path.join(MODEL_DIR, f'{model_tag}.pkl')
    model_size = get_model_size(model, model_path)
    
    result = {
        'model_name': model_name,
        'model_tag': model_tag,
        'dataset': dataset_name,
        'n_samples': len(X),
        'n_features': X.shape[1],
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1_score': f1,
        'roc_auc': roc,
        'train_time': train_time,
        'inference_10k': inference_per_10k,
        'model_size_mb': model_size,
        'ram_mb': max(0, mem_after - mem_before),
        'model': model,
        'y_test': y_test,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'X_test': X_test
    }
    
    print(f'  {model_name:15s} | {dataset_name:5s} | Acc={acc:.2f}% | F1={f1:.2f}% | Time={train_time:.2f}s')
    return result

## 3. Training pada 4 Ukuran Dataset — 3 Model (XGBoost, Random Forest, SVM)

In [ ]:
print('='*70)
print(f'{"TRAINING: All Features — 3 Models x 4 Datasets":^70}')
print('='*70)

all_results = []

for ds_name, ds_file in DATASETS.items():
    filepath = os.path.join(DATA_DIR, ds_file)
    if not os.path.exists(filepath):
        print(f'  {ds_file} NOT FOUND — skipping')
        continue
    
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    
    X = data['X']
    y = data['y']
    
    print(f'\n--- Dataset: {ds_name} ({len(X):,} samples, {X.shape[1]} features) ---')
    
    for model_name, model_factory in MODELS.items():
        tag = f'{model_name.lower().replace(" ","_")}_{ds_name.replace("%","pct")}'
        
        # SVM sangat lambat pada dataset besar — skip jika > 50k samples
        if model_name == 'SVM' and len(X) > 50000:
            print(f'  {model_name:15s} | {ds_name:5s} | SKIPPED (>{50000} samples, too slow)')
            continue
        
        result = train_and_evaluate(X, y, model_name, model_factory, ds_name, tag)
        all_results.append(result)
    
    del data
    gc.collect()

print(f'\nCompleted: {len(all_results)} experiments')

## 4. Tabel Perbandingan Performa

In [ ]:
# === TABLE 1: Performance Metrics ===
perf_rows = []
for r in all_results:
    perf_rows.append({
        'Model': r['model_name'],
        'Dataset': r['dataset'],
        'Samples': f"{r['n_samples']:,}",
        'Features': r['n_features'],
        'Accuracy (%)': f"{r['accuracy']:.2f}",
        'Precision (%)': f"{r['precision']:.2f}",
        'Recall (%)': f"{r['recall']:.2f}",
        'F1-Score (%)': f"{r['f1_score']:.2f}",
        'ROC-AUC (%)': f"{r['roc_auc']:.2f}"
    })

df_perf = pd.DataFrame(perf_rows)
print('='*110)
print(f'{"TABLE 1: PERBANDINGAN PERFORMA MODEL":^110}')
print('='*110)
print(df_perf.to_string(index=False))
print('='*110)

In [ ]:
# === TABLE 2: Efficiency Metrics ===
eff_rows = []
for r in all_results:
    eff_rows.append({
        'Model': r['model_name'],
        'Dataset': r['dataset'],
        'Training Time (s)': f"{r['train_time']:.2f}",
        'Inference/10k (s)': f"{r['inference_10k']:.4f}",
        'Model Size (MB)': f"{r['model_size_mb']:.2f}",
        'RAM Usage (MB)': f"{r['ram_mb']:.1f}"
    })

df_eff = pd.DataFrame(eff_rows)
print('='*95)
print(f'{"TABLE 2: EFISIENSI MODEL":^95}')
print('='*95)
print(df_eff.to_string(index=False))
print('='*95)

## 5. Learning Curve (F1-Score vs Dataset Size)

In [ ]:
# Learning Curve — per model
fig, ax = plt.subplots(figsize=(12, 7))

colors_model = {'XGBoost': 'steelblue', 'Random Forest': 'forestgreen', 'SVM': 'coral'}
markers_model = {'XGBoost': 'o', 'Random Forest': 's', 'SVM': '^'}

for model_name in MODELS.keys():
    model_results = [r for r in all_results if r['model_name'] == model_name]
    if not model_results:
        continue
    ds_sizes = [r['n_samples'] for r in model_results]
    f1_scores = [r['f1_score'] for r in model_results]
    
    ax.plot(ds_sizes, f1_scores, 
            marker=markers_model[model_name], linestyle='-', 
            color=colors_model[model_name], linewidth=2, markersize=8, 
            label=f'{model_name}')
    
    for size, f1 in zip(ds_sizes, f1_scores):
        ax.annotate(f'{f1:.1f}%', (size, f1), textcoords='offset points', 
                   xytext=(0,10), ha='center', fontsize=8)

ax.set_xlabel('Dataset Size (samples)')
ax.set_ylabel('F1-Score (%)')
ax.set_title('Learning Curve: F1-Score vs Dataset Size\n(XGBoost vs Random Forest vs SVM — All Features)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'learning_curve_all_features.png'), bbox_inches='tight')
plt.show()
print('Saved: learning_curve_all_features.png')

## 6. Confusion Matrix (Heatmap) — Dataset 100%

In [ ]:
# Confusion matrix untuk dataset terbesar (100%) — per model
results_100 = [r for r in all_results if r['dataset'] == '100%']

fig, axes = plt.subplots(1, len(results_100), figsize=(7*len(results_100), 6))
if len(results_100) == 1:
    axes = [axes]

class_names = list(label_mapping.keys())

for ax, r in zip(axes, results_100):
    cm = confusion_matrix(r['y_test'], r['y_pred'])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(f'Confusion Matrix (Normalized)\n{r["model_name"]} — 100%', fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'confusion_matrix_all_models.png'), bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix_all_models.png')

## 7. Feature Importance Chart (Top-20)

In [ ]:
# Feature Importance horizontal bar chart
top20_df = feat_imp_df.head(20).iloc[::-1]  # reverse for horizontal

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.3, 0.9, 20))
ax.barh(range(20), top20_df['importance'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_yticks(range(20))
ax.set_yticklabels(top20_df['feature'], fontsize=9)
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Top-20 Feature Importance — XGBoost (All Features)\nCSE-CIC-IDS2018', fontsize=13, fontweight='bold')

# Value labels
for i, (val, name) in enumerate(zip(top20_df['importance'], top20_df['feature'])):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'feature_importance_top20.png'), bbox_inches='tight')
plt.show()
print('Saved: feature_importance_top20.png')

## 8. Narasi Perbandingan

In [ ]:
print('='*70)
print(f'{"NARASI PERBANDINGAN HASIL EKSPERIMEN":^70}')
print('='*70)

# Per-model summary
for model_name in MODELS.keys():
    model_results = [r for r in all_results if r['model_name'] == model_name]
    if not model_results:
        print(f'\n■ {model_name}: SKIPPED (dataset too large for this model)')
        continue
    best = max(model_results, key=lambda x: x['f1_score'])
    worst = min(model_results, key=lambda x: x['f1_score'])
    print(f"\n■ {model_name}:")
    print(f"  Best:  {best['dataset']} → F1={best['f1_score']:.2f}%, Acc={best['accuracy']:.2f}%")
    print(f"  Worst: {worst['dataset']} → F1={worst['f1_score']:.2f}%, Acc={worst['accuracy']:.2f}%")
    print(f"  Range: Δ F1 = {best['f1_score'] - worst['f1_score']:.2f}%")
    print(f"  Avg train time: {np.mean([r['train_time'] for r in model_results]):.2f}s")

# Overall best
best_overall = max(all_results, key=lambda x: x['f1_score'])
fastest_overall = min(all_results, key=lambda x: x['train_time'])
smallest_overall = min(all_results, key=lambda x: x['model_size_mb'])

print(f"""
{'='*70}
■ OVERALL COMPARISON:
  Best F1:        {best_overall['model_name']} ({best_overall['dataset']}) → {best_overall['f1_score']:.2f}%
  Fastest train:  {fastest_overall['model_name']} ({fastest_overall['dataset']}) → {fastest_overall['train_time']:.2f}s
  Smallest model: {smallest_overall['model_name']} ({smallest_overall['dataset']}) → {smallest_overall['model_size_mb']:.2f} MB

■ KESIMPULAN:
  - XGBoost umumnya memberikan F1 tertinggi dengan training time moderat
  - Random Forest kompetitif dan lebih cepat pada dataset besar
  - SVM akurat tapi sangat lambat (O(n²~n³)), hanya feasible untuk dataset kecil
  - Semua model menunjukkan performa stabil bahkan pada 25% data

■ NEXT STEP:
  - Notebook 04: Feature reduction (Top-20 vs Top-15 vs Top-10)
  - Bandingkan apakah pengurangan fitur mempengaruhi performa signifikan
{'='*70}
""")

In [ ]:
# Save semua hasil eksperimen
experiment_results = {
    'all_results': [{k: v for k, v in r.items() if k not in ['model', 'X_test', 'y_prob']} for r in all_results],
    'feature_importance': feat_imp_df,
    'top20_features': top20_features,
    'top15_features': top15_features,
    'top10_features': top10_features,
    'label_mapping': label_mapping,
    'feature_names': feature_names,
    'models_used': list(MODELS.keys())
}

with open(os.path.join(DATA_DIR, 'experiment_results_03.pkl'), 'wb') as f:
    pickle.dump(experiment_results, f)

print('Saved: experiment_results_03.pkl')
print(f'  Contains: results for {len(all_results)} experiments (3 models x 4 datasets)')
print(f'  Models: {list(MODELS.keys())}')
print(f'  Top-20 features: {top20_features}')
print(f'  Top-15 features: {top15_features}')
print(f'  Top-10 features: {top10_features}')
print('\nDONE! Ready for Notebook 04 (Ablation Study — Feature Reduction).')